# Notebook Kiểm Thử Offline Thuật Toán Nhận Diện Làn Đường JetRacer

Notebook này giúp bạn kiểm thử offline thuật toán xử lý ảnh HSV + Quét hàng (Scanline) trên các ảnh tĩnh hoặc video ghi lại từ camera của xe mà không cần chạy ROS.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Thêm thư mục gốc của dự án vào sys.path để import các module trong src
repo_root = os.path.dirname(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.config import settings
from src.perception.camera.camera_processor import CameraProcessor
from src.core.blackboard import Blackboard

## 1. Khởi tạo Camera Processor

In [ ]:
blackboard = Blackboard()
camera = CameraProcessor(blackboard)
camera.initialize()
print("Chiều rộng làn đường ước lượng mặc định:", camera.estimated_lane_width)

## 2. Tải ảnh test thực tế hoặc tạo ảnh giả lập

In [ ]:
# Tạo một ảnh màu xanh lá/xanh dương giả lập đường tối và vạch biên màu cam để test nếu không có ảnh thực tế
test_image_path = "test_lane.jpg"

if os.path.exists(test_image_path):
    frame = cv2.imread(test_image_path)
    print(f"Đã đọc ảnh thực tế: {test_image_path} (Kích thước: {frame.shape})")
else:
    # Tạo ảnh giả lập 300x300 lòng đường xám đậm, bên ngoài trắng, biên đỏ
    frame = np.ones((300, 300, 3), dtype=np.uint8) * 255 # Nền trắng ngoài đường
    # Lòng đường nhựa màu đen/xám (cột 60 đến 240)
    frame[:, 60:240] = [40, 50, 40] # Tông xanh xám tối
    # Vạch ranh giới màu đỏ (rộng 5px ở cột 58-62 và 238-242)
    frame[:, 58:63] = [0, 0, 200] # BGR Red
    frame[:, 237:242] = [0, 0, 200]
    print("Không tìm thấy ảnh test thực tế → Tạo ảnh giả lập để chạy thử.")

# Hiển thị ảnh gốc
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("Original Test Image")
plt.axis('off')
plt.show()

## 3. Chạy qua bộ xử lý ảnh của Robot

In [ ]:
# Chạy thuật toán process_frame
center_x_bottom, waypoints, debug_img = camera.process_frame(frame, dodge_direction=0.0)

print(f"Tọa độ tâm đường sát mũi xe (x_bottom): {center_x_bottom}")
print(f"Danh sách Waypoints quét được: {waypoints}")

## 4. Trực quan hóa kết quả đầu ra

In [ ]:
# Vẽ các điểm waypoints lên ảnh gốc để xem độ bám đường
output_img = frame.copy()
for pt in waypoints:
    cv2.circle(output_img, pt, 5, (0, 255, 255), -1) # Chấm vàng tâm làn
if len(waypoints) >= 2:
    pts = np.array(waypoints, np.int32).reshape((-1, 1, 2))
    cv2.polylines(output_img, [pts], False, (0, 0, 255), 2) # Nối đỏ

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Trực quan hóa ảnh kết quả
axes[0].imshow(cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Lane Tracking Waypoints")
axes[0].axis('off')

# Trực quan hóa ảnh Canny sạch + ROI
axes[1].imshow(cv2.cvtColor(debug_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Canny Edges & ROI Mask")
axes[1].axis('off')

plt.tight_layout()
plt.show()